unzip the nii.gz
list all t1w files to insert to the cat12 runner script

In [ ]:
from pathlib import Path
import os
import re
from tqdm import tqdm
import pandas as pd
import json

# unzipping nifti for cat12 analysis

In [ ]:
# source_path = Path("/home/gaia/Projects/HardDriveOutput/BIDS_over_thirties") #EDIT THIS PATH AS NEEDED


# # unzipping the niftii after conversion from dicom 
# gzipped_files = list(source_path.rglob('*.nii.gz'))

# for gzipped in tqdm(gzipped_files, desc="Decompressing NIfTI files"):
#     gunzipped = gzipped.with_suffix('').with_suffix('.nii')
    
#     # Skip if the .nii file already exists
#     if gunzipped.exists():
#         # Optional: print skipped file or just continue silently
#         # print(f"Skipping {gunzipped}, already exists.")
#         continue
    
#     cmd = f"gunzip -c '{gzipped}' > '{gunzipped}'"
#     os.system(cmd)

# create the list for cat12 runner script

In [ ]:
import os
import re
import json

def list_nifti_by_protocol(root_dir, protocol, output_file=None):
    """
    Lists NIfTI files ending with a given protocol pattern (e.g., _T1w.nii),
    excluding:
      - Files in 'derivatives' folders
      - Files already processed by CAT12 (if any CAT12 output exists for the same subject-session)

    Parameters:
        root_dir (str): Root directory to search.
        protocol (str): Protocol suffix to match, e.g., 'T1w', 'FLAIR', 'dwi'.
        output_file (str): Optional output file. Defaults to '{protocol}_list.txt'.
    """
    if output_file is None:
        output_file = f"{protocol}_list.txt"

    nii_files = []
    found_participant_ids = set()
    processed_pairs = set()

    # 1. Collect subject-session pairs already processed by CAT12
    derivatives_root = os.path.join(root_dir, 'derivatives', 'CAT12.9_2577')
    if os.path.exists(derivatives_root):
        for dirpath, _, files in os.walk(derivatives_root):
            for file in files:
                match = re.match(r".*sub-(ls\d+)_ses-(\d+)_.*", file)
                if match:
                    processed_pairs.add(match.groups())

    # 2. Scan for NIfTI files and apply filters
    for root_dirpath, _, files in os.walk(root_dir):
        if 'derivatives' in root_dirpath.split(os.sep):  # Skip 'derivatives' folders
            continue

        for file in files:
            # Check protocol suffix
            if not file.endswith(f"_{protocol}.nii"):
                continue

            # Parse subject and session from filename
            match = re.match(r"sub-(ls\d+)_ses-(\d+)_.*", file)
            if not match:
                continue  # Skip files not matching the subject-session pattern

            subject, session = match.groups()

            # Apply CAT12 processed exclusion
            if (subject, session) in processed_pairs:
                continue

            full_path = os.path.join(root_dirpath, file)
            nii_files.append(full_path)
            found_participant_ids.add(subject)

    # 3. Write results to file
    with open(output_file, 'w') as f:
        for path in nii_files:
            f.write(f"'{path}'\n")

    print(f"Found {len(nii_files)} *_{protocol}.nii files (excluding processed and derivatives).")
    print(f"Found {len(found_participant_ids)} unique participants.")
    print(f"List saved to: {output_file}")


# Example usage:
list_nifti_by_protocol('path/to/list', 'T1w') # EDIT THIS PATH AS NEEDED
